# 04. Relational Model & ER Model

*Module: 1 — Basics · Companion summary: `lectures/04-database-concepts-relational-model.md` · SQL engine: SQLite (in-memory)*

## What you will study

- map entities → tables, attributes → columns, values → rows
- choose CHAR vs VARCHAR-style types for the job
- see primary keys and foreign keys enforce relationships

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

## 📗 ER → relational mapping

| ER concept | Relational result | Library example |
|---|---|---|
| Entity (rectangle in a diagram) | **Table** | AUTHOR, BOOK |
| Attribute (oval) | **Column** | title, year, lastname |
| Data values | **Rows** | ('A1', 'Chong', ...) |
| Identifier | **Primary key** | book_id |
| Attribute referencing another table's PK | **Foreign key** | book.author_id |

Types: `INTEGER` / `DECIMAL` for numbers, `DATE`/timestamp types for dates, and for
strings: fixed-length codes get CHAR-style types, human-entered text gets VARCHAR-style.
(SQLite only formalizes INTEGER / REAL / TEXT / BLOB / NULL — sizes are advisory.)

## 🧪 Build the library mini-model — BOOK references AUTHOR

In [ ]:
q("""
CREATE TABLE author (
    author_id TEXT PRIMARY KEY,     -- equivalent of CHAR(2) but sized freely
    lastname  TEXT NOT NULL,
    firstname TEXT NOT NULL,
    email TEXT, city TEXT, country TEXT
)
""")
q("""
CREATE TABLE book (
    book_id  TEXT PRIMARY KEY,
    title    TEXT,                  -- VARCHAR-style: varies per book
    pages    INTEGER,
    author_id TEXT REFERENCES author(author_id)
)
""")

## 🧪 PRIMARY KEY enforces uniqueness; FOREIGN KEY enforces the link

In [ ]:
q("INSERT INTO author VALUES ('A1', 'Chong', 'Raul', null, 'Toronto', 'CA')")
try:
    # duplicate primary key — the db refuses
    q("INSERT INTO author VALUES ('A1', 'Faker', 'Fake', null, null, null)")
except sqlite3.IntegrityError as e:
    print('PRIMARY KEY refused a duplicate:', repr(e))
try:
    # a book pointing at an author that does not exist
    q("INSERT INTO book VALUES ('B1', 'A Book', 200, 'A9')")
except sqlite3.IntegrityError as e:
    print('FOREIGN KEY refused an unknown author:', repr(e))
q("INSERT INTO book VALUES ('B1', 'The Little Prince', 292, 'A1')")  # now valid
q("SELECT * FROM book")

## 🧪 Inspect the model you built (a taste of lecture 25)

In [ ]:
q("SELECT name, type FROM sqlite_master WHERE type IN ('table')")

## 💡 Tips & tricks

- Sketch the ERD first, SQL second — joins are transcribed from PK→FK links (lecture 18).
- `CHAR`-style for truly fixed lengths (country codes, ISBN), `VARCHAR`-style for titles,
  names, emails.

## 🎯 Your turn

1. Add entity `BORROWER (borrower_id, lastname, country)` yourself.
2. Add entity `LOAN (borrower_id, book_id, loan_date)` whose two keys reference the
   other tables — then try inserting a loan for an unknown borrower to see the FK fire.

In [ ]:
# 💡 Solution
q("CREATE TABLE borrower (borrower_id TEXT PRIMARY KEY, lastname TEXT, country TEXT)")
q("CREATE TABLE loan (borrower_id TEXT REFERENCES borrower(borrower_id), book_id TEXT REFERENCES book(book_id), loan_date TEXT)")
q("INSERT INTO borrower VALUES ('B1', 'Ali', 'IN')")
try:
    q("INSERT INTO loan VALUES ('B99', 'B1', '2024-01-01')")
except sqlite3.IntegrityError as e:
    print('foreign key enforced:', repr(e))
q("INSERT INTO loan VALUES ('B1', 'B1', '2024-01-01')")
q("SELECT * FROM loan")